# P1 Chunking benchmark (Variant): NFCorpus and a HotpotQA subset

Compares **fixed-length, recursive and semantic chunking** at two chunk sizes (128 and 256 tokens)
against a no-chunking baseline, with the BGE-small embedder. Reports Recall@5, Recall@10, MRR@10 and NDCG@10
(T1), chunk counts and indexing time (T2), NDCG@10 bar chart (F1) and Recall@K curves K = 1..20 (F2).

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then Runtime > Run all (or run cells in order).
No uploads needed; everything is written by the cells below. Expected run time: roughly 10-20 minutes on a T4
(I have not timed it on Colab). Results are zipped and downloaded by the last cell.

## 1. Check the GPU and install packages

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q sentence-transformers datasets tabulate matplotlib pandas

## 2. Write the project code (p1 package)

In [ ]:
import os
os.makedirs('/content/p1proj/p1', exist_ok=True)
%cd /content/p1proj
open('p1/__init__.py','w').write('')

In [ ]:
%%writefile p1/config.py
"""Configuration for P1 (Variant): chunking benchmark on NFCorpus and a HotpotQA subset.

Environment overrides: P1_ROOT (output root), P1_SMOKE=1 (tiny run for testing only).
"""
import os
from pathlib import Path

ROOT = Path(os.environ.get("P1_ROOT", Path(__file__).resolve().parents[1]))
RESULTS = ROOT / "results"

SMOKE = os.environ.get("P1_SMOKE") == "1"

# ---- models (open, small) -----------------------------------------------
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # BGE retrieval instruction
MAX_SEQ = 512

# ---- experiment grid -----------------------------------------------------
DATASETS = ["nfcorpus", "hotpotqa"]
STRATEGIES = ["fixed", "recursive", "semantic"]
CHUNK_SIZES = [128, 256]            # max tokens per chunk (BGE tokenizer tokens)
CHUNK_OVERLAP = 0                   # tokens; kept 0 so strategies differ only in where they cut
PREPEND_TITLE = False               # True = prefix every chunk with the document title
BASELINE = "whole_doc"              # no chunking: whole document, truncated to MAX_SEQ by the model

# semantic chunking: cut where adjacent-sentence distance exceeds this corpus-wide percentile,
# subject to the max chunk size and a minimum chunk size (fraction of max)
SEMANTIC_PERCENTILE = 80
SEMANTIC_MIN_FRAC = 0.25

# ---- data ----------------------------------------------------------------
SEEDS = [0, 1, 2]                   # HotpotQA: three different question samples. NFCorpus is deterministic (one run).
HOTPOT_N_QUESTIONS = 500            # questions per seed; corpus = their context paragraphs
NFCORPUS_MAX_QUERIES = None         # None = all test queries with judgments (323)

# ---- evaluation ----------------------------------------------------------
KS = list(range(1, 21))             # recall curve K = 1..20
DEPTH = 100                         # ranked documents kept per query
BOOTSTRAP = 1000

if SMOKE:                           # never use for reported numbers
    SEEDS = [0]
    HOTPOT_N_QUESTIONS = 60
    NFCORPUS_MAX_QUERIES = 40
    BOOTSTRAP = 50

In [ ]:
%%writefile p1/data.py
"""Dataset loaders. Each returns a dict:
    corpus : {doc_id: {"title": str, "text": str}}
    queries: {qid: str}
    qrels  : {qid: {doc_id: graded relevance (>0)}}
    qtype  : {qid: str}   (question category, may be empty)
"""
import random

from datasets import load_dataset

from . import config as C


def load_nfcorpus(max_queries=None):
    corpus_ds = load_dataset("BeIR/nfcorpus", "corpus")["corpus"]
    query_ds = load_dataset("BeIR/nfcorpus", "queries")["queries"]
    qrel_ds = load_dataset("BeIR/nfcorpus-qrels")["test"]

    qrels = {}
    for r in qrel_ds:
        if r["score"] > 0:
            qrels.setdefault(r["query-id"], {})[r["corpus-id"]] = int(r["score"])
    queries = {q["_id"]: q["text"] for q in query_ds if q["_id"] in qrels}
    qids = sorted(queries)
    if max_queries:
        qids = qids[:max_queries]
    corpus = {d["_id"]: {"title": d["title"] or "", "text": d["text"] or ""} for d in corpus_ds}
    qrels = {q: {d: s for d, s in qrels[q].items() if d in corpus} for q in qids}
    qids = [q for q in qids if qrels[q]]
    if C.SMOKE:        # testing only: keep relevant docs plus 300 distractors so CPU runs finish quickly
        keep = {d for q in qids for d in qrels[q]}
        extra = [d for d in sorted(corpus) if d not in keep][:300]
        corpus = {d: corpus[d] for d in sorted(keep) + extra}
    return {"corpus": corpus, "queries": {q: queries[q] for q in qids},
            "qrels": {q: qrels[q] for q in qids}, "qtype": {q: "" for q in qids}}


def load_hotpotqa(seed, n_questions):
    """Random sample of HotpotQA (distractor setting, validation split).

    Corpus = every context paragraph of the sampled questions (gold + distractors), de-duplicated
    by Wikipedia title. Relevant documents = the paragraphs named in `supporting_facts`.
    """
    ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
    idx = list(range(len(ds)))
    random.Random(seed).shuffle(idx)
    corpus, queries, qrels, qtype = {}, {}, {}, {}
    for i in idx[:n_questions]:
        ex = ds[i]
        for title, sents in zip(ex["context"]["title"], ex["context"]["sentences"]):
            corpus.setdefault(title, {"title": title, "text": "".join(sents).strip()})
        qid = ex["id"]
        queries[qid] = ex["question"]
        qrels[qid] = {t: 1 for t in set(ex["supporting_facts"]["title"]) if t in corpus}
        qtype[qid] = ex["type"]                       # bridge | comparison
    return {"corpus": corpus, "queries": queries, "qrels": qrels, "qtype": qtype}


def load(dataset, seed=0):
    if dataset == "nfcorpus":
        return load_nfcorpus(C.NFCORPUS_MAX_QUERIES)
    if dataset == "hotpotqa":
        return load_hotpotqa(seed, C.HOTPOT_N_QUESTIONS)
    raise ValueError(dataset)


def doc_text(d):
    """Text that gets chunked: title and body, as in BEIR's title + text convention."""
    return f"{d['title']}\n{d['text']}".strip() if d["title"] else d["text"]

In [ ]:
%%writefile p1/chunkers.py
"""Chunking strategies. All sizes are measured in tokens of the embedder's own tokenizer
(BGE-small, uncased WordPiece), and all chunks are slices of the original text.

    whole_doc : baseline, no chunking (the model truncates at 512 tokens)
    fixed     : consecutive windows of `size` tokens (optional overlap)
    recursive : split on paragraph, line, sentence, clause, word boundaries until pieces fit,
                then merge neighbouring pieces up to `size`
    semantic  : sentences grouped until adjacent-sentence similarity drops (corpus-wide
                percentile threshold) or `size` is reached
"""
import re
import time

import numpy as np

from . import config as C


class Tok:
    """Thin wrapper around the embedder tokenizer."""

    def __init__(self, name=C.EMBED_MODEL):
        from transformers import AutoTokenizer
        self.t = AutoTokenizer.from_pretrained(name)
        self.t.model_max_length = 10 ** 9           # we chunk long docs ourselves; silence length warnings

    def count(self, text):
        return len(self.t(text, add_special_tokens=False)["input_ids"])

    def counts(self, texts):
        if not texts:
            return []
        enc = self.t(texts, add_special_tokens=False)["input_ids"]
        return [len(x) for x in enc]

    def offsets(self, text):
        enc = self.t(text, add_special_tokens=False, return_offsets_mapping=True)
        return enc["offset_mapping"]


# ---- fixed-length ----------------------------------------------------------
def chunk_fixed(text, size, tok, overlap=0):
    off = tok.offsets(text)
    if not off:
        return []
    step = max(size - overlap, 1)
    out, i = [], 0
    while i < len(off):
        j = min(i + size, len(off))
        piece = text[off[i][0]:off[j - 1][1]].strip()
        if piece:
            out.append(piece)
        if j == len(off):
            break
        i += step
    return out


# ---- recursive -------------------------------------------------------------
SEPARATORS = ("\n\n", "\n", ". ", "? ", "! ", "; ", ", ", " ")


def chunk_recursive(text, size, tok, overlap=0):
    def split(t, seps):
        if tok.count(t) <= size:
            return [t]
        if not seps:
            return chunk_fixed(t, size, tok)          # last resort: no separator left
        sep, rest = seps[0], seps[1:]
        parts = t.split(sep)
        if len(parts) == 1:
            return split(t, rest)
        pieces = []
        for i, p in enumerate(parts):
            p = p + (sep if i < len(parts) - 1 else "")      # keep the separator so text is preserved
            pieces.extend(split(p, rest) if tok.count(p) > size else [p])
        return pieces

    pieces = split(text, SEPARATORS)
    counts = tok.counts(pieces)
    chunks, cur, cur_n = [], "", 0
    for p, n in zip(pieces, counts):
        if cur and cur_n + n > size:
            chunks.append(cur)
            cur, cur_n = p, n
        else:
            cur, cur_n = cur + p, cur_n + n
    if cur:
        chunks.append(cur)
    return [c.strip() for c in chunks if c.strip()]


# ---- semantic --------------------------------------------------------------
_SENT = re.compile(r'(?<=[.!?])\s+(?=[A-Z0-9"\'(\[])|\n+')


def split_sentences(text):
    return [s.strip() for s in _SENT.split(text) if s and s.strip()]


class SemanticState:
    """Sentence embeddings for one corpus, computed once and shared by both chunk sizes.
    The time it took is added to every semantic run's chunking time so cost stays honest."""

    def __init__(self, docs, tok, embedder):
        t0 = time.time()
        self.sents = {d: split_sentences(t) for d, t in docs.items()}
        flat = [s for d in self.sents for s in self.sents[d]]
        emb = embedder.encode_texts(flat)                      # (n_sent, dim) normalised, torch
        self.ntok = dict(zip(range(len(flat)), tok.counts(flat)))
        self.dist, pos = {}, 0
        all_d = []
        for d in self.sents:
            n = len(self.sents[d])
            e = emb[pos:pos + n]
            if n > 1:
                dd = (1 - (e[:-1] * e[1:]).sum(1)).cpu().numpy()
                all_d.append(dd)
            else:
                dd = np.zeros(0)
            self.dist[d] = (pos, dd)
            pos += n
        self.threshold = float(np.percentile(np.concatenate(all_d), C.SEMANTIC_PERCENTILE)) if all_d else 1.0
        self.seconds = time.time() - t0


def chunk_semantic(doc_id, state, size, tok, overlap=0):
    sents = state.sents[doc_id]
    start, dist = state.dist[doc_id]
    min_tokens = max(int(size * C.SEMANTIC_MIN_FRAC), 1)
    chunks, cur, cur_n = [], [], 0

    def flush():
        nonlocal cur, cur_n
        if cur:
            chunks.append(" ".join(cur))
        cur, cur_n = [], 0

    for i, s in enumerate(sents):
        n = state.ntok[start + i]
        if n > size:                                   # one sentence longer than a chunk: hard split
            flush()
            chunks.extend(chunk_fixed(s, size, tok))
            continue
        if cur and (cur_n + n > size or (i > 0 and dist[i - 1] > state.threshold and cur_n >= min_tokens)):
            flush()
        cur.append(s)
        cur_n += n
    flush()
    return chunks


# ---- driver ----------------------------------------------------------------
def build_chunks(corpus_docs, titles, strategy, size, tok, sem_state=None):
    """corpus_docs: {doc_id: text}. Returns (chunk_texts, chunk_doc_ids, seconds spent chunking)."""
    t0 = time.time()
    texts, owners = [], []
    for d, text in corpus_docs.items():
        if strategy == "whole_doc":
            pieces = [text]
        elif strategy == "fixed":
            pieces = chunk_fixed(text, size, tok, C.CHUNK_OVERLAP)
        elif strategy == "recursive":
            pieces = chunk_recursive(text, size, tok, C.CHUNK_OVERLAP)
        elif strategy == "semantic":
            pieces = chunk_semantic(d, sem_state, size, tok, C.CHUNK_OVERLAP)
        else:
            raise ValueError(strategy)
        if not pieces:
            pieces = [text or " "]
        if C.PREPEND_TITLE and strategy != "whole_doc" and titles.get(d):
            pieces = [p if i == 0 else f"{titles[d]}\n{p}" for i, p in enumerate(pieces)]
        texts.extend(pieces)
        owners.extend([d] * len(pieces))
    seconds = time.time() - t0
    if strategy == "semantic" and sem_state is not None:
        seconds += sem_state.seconds
    return texts, owners, seconds

In [ ]:
%%writefile p1/metrics.py
"""Ranking metrics at document level (chunk hits are collapsed to their parent document)."""
import math

import numpy as np


def recall_at_k(ranked, rel, k):
    return len(set(ranked[:k]) & set(rel)) / len(rel)


def mrr_at_k(ranked, rel, k=10):
    for i, d in enumerate(ranked[:k], 1):
        if d in rel:
            return 1.0 / i
    return 0.0


def ndcg_at_k(ranked, rel, k=10):
    """Linear gain (gain = graded relevance), log2 discount, as in BEIR / trec_eval ndcg_cut."""
    dcg = sum(rel.get(d, 0) / math.log2(i + 1) for i, d in enumerate(ranked[:k], 1))
    ideal = sorted(rel.values(), reverse=True)[:k]
    idcg = sum(g / math.log2(i + 1) for i, g in enumerate(ideal, 1))
    return dcg / idcg if idcg > 0 else 0.0


def bootstrap_ci(values, n=1000, seed=0):
    v = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    means = [v[rng.integers(0, len(v), len(v))].mean() for _ in range(n)]
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


if __name__ == "__main__":      # self-test against hand-computed values
    rel = {"a": 2, "b": 1, "c": 1}
    r = ["x", "a", "y", "b"]
    assert abs(recall_at_k(r, rel, 4) - 2 / 3) < 1e-9
    assert mrr_at_k(r, rel) == 0.5
    dcg = 2 / math.log2(3) + 1 / math.log2(5)
    idcg = 2 / math.log2(2) + 1 / math.log2(3) + 1 / math.log2(4)
    assert abs(ndcg_at_k(r, rel) - dcg / idcg) < 1e-9
    assert ndcg_at_k(["a", "b", "c"], rel) == 1.0
    print("metrics self-test OK")

In [ ]:
%%writefile p1/run_experiment.py
"""Run the whole chunking benchmark:  python -m p1.run_experiment

For every (dataset, seed): build chunks with each strategy x chunk size (plus the no-chunking
baseline), embed them with BGE-small, retrieve, collapse chunk hits to documents (best chunk
score per document) and score Recall@1..20, MRR@10 and NDCG@10.

Writes to results/: results.csv, per_query.csv.gz, index_cost.csv, doc_stats.csv, run_info.json
"""
import json
import platform
import random
import time

import numpy as np
import pandas as pd
import torch

from . import config as C
from . import data as D
from .chunkers import SemanticState, Tok, build_chunks
from .metrics import mrr_at_k, ndcg_at_k, recall_at_k


class Embedder:
    def __init__(self):
        from sentence_transformers import SentenceTransformer
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(C.EMBED_MODEL, device=self.device)
        self.model.max_seq_length = C.MAX_SEQ
        self.encode_texts(["warm up the model"] * 8)          # keep CUDA start-up out of the timings

    def encode_texts(self, texts, batch=128):
        e = self.model.encode(texts, batch_size=batch, normalize_embeddings=True,
                              convert_to_tensor=True, show_progress_bar=False)
        if self.device == "cuda":
            torch.cuda.synchronize()
        return e

    def encode_queries(self, queries):
        return self.encode_texts([C.QUERY_PREFIX + q for q in queries])


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def rank_documents(qemb, cemb, owner_idx, n_docs, depth):
    """Best-chunk-per-document ranking. Returns (Q, depth) tensor of document indices."""
    scores = qemb @ cemb.T                                           # (Q, n_chunks)
    doc_scores = torch.full((scores.shape[0], n_docs), -1e9, device=scores.device)
    doc_scores.scatter_reduce_(1, owner_idx.unsqueeze(0).expand_as(scores), scores, reduce="amax")
    return doc_scores.topk(min(depth, n_docs), dim=1).indices.cpu().numpy()


def main():
    C.RESULTS.mkdir(parents=True, exist_ok=True)
    tok, emb = Tok(), Embedder()
    results, per_query, costs, docstats = [], [], [], []

    for ds_name in C.DATASETS:
        seeds = C.SEEDS if ds_name == "hotpotqa" else [0]
        for seed in seeds:
            seed_all(seed)
            ds = D.load(ds_name, seed)
            doc_ids = list(ds["corpus"])
            docs = {d: D.doc_text(ds["corpus"][d]) for d in doc_ids}
            titles = {d: ds["corpus"][d]["title"] for d in doc_ids}
            owner_of = {d: i for i, d in enumerate(doc_ids)}
            qids = list(ds["queries"])
            qemb = emb.encode_queries([ds["queries"][q] for q in qids])
            print(f"\n== {ds_name} seed {seed}: {len(doc_ids)} docs, {len(qids)} queries ==", flush=True)

            dtok = np.array(tok.counts([docs[d] for d in doc_ids]))
            docstats.append({"dataset": ds_name, "seed": seed, "n_docs": len(doc_ids), "n_queries": len(qids),
                             "mean_doc_tokens": dtok.mean(), "median_doc_tokens": float(np.median(dtok)),
                             **{f"pct_docs_over_{s}": float((dtok > s).mean() * 100) for s in C.CHUNK_SIZES + [512]}})

            sem_state = SemanticState(docs, tok, emb) if "semantic" in C.STRATEGIES else None
            grid = [(C.BASELINE, 0)] + [(s, z) for s in C.STRATEGIES for z in C.CHUNK_SIZES]
            for strategy, size in grid:
                texts, owners, chunk_s = build_chunks(docs, titles, strategy, size, tok, sem_state)
                t0 = time.time()
                cemb = emb.encode_texts(texts)
                embed_s = time.time() - t0
                owner_idx = torch.tensor([owner_of[o] for o in owners], device=cemb.device)
                ranked = rank_documents(qemb, cemb, owner_idx, len(doc_ids), C.DEPTH)
                ntok = np.array(tok.counts(texts))
                costs.append({"dataset": ds_name, "seed": seed, "strategy": strategy, "chunk_size": size,
                              "n_docs": len(doc_ids), "n_chunks": len(texts), "mean_chunk_tokens": ntok.mean(),
                              "chunk_seconds": chunk_s, "embed_seconds": embed_s,
                              "index_seconds": chunk_s + embed_s})

                curves, mrrs, ndcgs = [], [], []
                for qi, q in enumerate(qids):
                    rel = ds["qrels"][q]
                    r = [doc_ids[j] for j in ranked[qi]]
                    rec = [recall_at_k(r, rel, k) for k in C.KS]
                    m, n = mrr_at_k(r, rel, 10), ndcg_at_k(r, rel, 10)
                    curves.append(rec); mrrs.append(m); ndcgs.append(n)
                    per_query.append({"dataset": ds_name, "seed": seed, "strategy": strategy, "chunk_size": size,
                                      "qid": q, "qtype": ds["qtype"][q], "n_relevant": len(rel),
                                      "recall@5": rec[4], "recall@10": rec[9], "mrr@10": m, "ndcg@10": n,
                                      "top10": "|".join(r[:10]), "relevant": "|".join(rel)})
                curve = np.mean(curves, axis=0)
                base = {"dataset": ds_name, "seed": seed, "strategy": strategy, "chunk_size": size, "n_queries": len(qids)}
                for k, v in zip(C.KS, curve):
                    results.append({**base, "metric": f"recall@{k}", "value": float(v)})
                results.append({**base, "metric": "mrr@10", "value": float(np.mean(mrrs))})
                results.append({**base, "metric": "ndcg@10", "value": float(np.mean(ndcgs))})
                for mname in ("n_chunks", "index_seconds"):
                    results.append({**base, "metric": mname, "value": float(costs[-1][mname])})
                print(f"{strategy:9s} size {size:3d}  chunks {len(texts):6d}  R@5 {curve[4]:.3f}  R@10 {curve[9]:.3f}  "
                      f"MRR {np.mean(mrrs):.3f}  NDCG@10 {np.mean(ndcgs):.3f}  index {chunk_s + embed_s:.1f}s", flush=True)
                del cemb
                if emb.device == "cuda":
                    torch.cuda.empty_cache()

    pd.DataFrame(results).to_csv(C.RESULTS / "results.csv", index=False)
    pd.DataFrame(per_query).to_csv(C.RESULTS / "per_query.csv.gz", index=False)
    pd.DataFrame(costs).to_csv(C.RESULTS / "index_cost.csv", index=False)
    pd.DataFrame(docstats).to_csv(C.RESULTS / "doc_stats.csv", index=False)
    import sentence_transformers, transformers, datasets
    info = {"embed_model": C.EMBED_MODEL, "device": emb.device,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "python": platform.python_version(), "torch": torch.__version__,
            "sentence_transformers": sentence_transformers.__version__, "transformers": transformers.__version__,
            "datasets": datasets.__version__, "smoke": C.SMOKE,
            "config": {k: getattr(C, k) for k in ("DATASETS", "STRATEGIES", "CHUNK_SIZES", "CHUNK_OVERLAP",
                                                  "PREPEND_TITLE", "SEMANTIC_PERCENTILE", "SEMANTIC_MIN_FRAC",
                                                  "SEEDS", "HOTPOT_N_QUESTIONS", "NFCORPUS_MAX_QUERIES", "DEPTH")}}
    (C.RESULTS / "run_info.json").write_text(json.dumps(info, indent=2))
    print("\nwrote results to", C.RESULTS)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile p1/report.py
"""Build the report tables (T1, T2) and figures (F1, F2) from results/:  python -m p1.report

Outputs in results/: T1_quality.csv, T2_cost.csv, T3_ndcg_ci.csv, tables.md,
F1_ndcg_by_strategy.png, F2_recall_curves.png, F3_hotpot_by_qtype.png,
failure_cases.csv, doc_type_summary.csv
"""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from . import config as C
from .metrics import bootstrap_ci

COL = {"fixed": "#1D9E75", "recursive": "#534AB7", "semantic": "#D85A30", "whole_doc": "#888780"}
NAME = {"fixed": "Fixed-length", "recursive": "Recursive", "semantic": "Semantic", "whole_doc": "No chunking (baseline)"}
DS_NAME = {"nfcorpus": "NFCorpus", "hotpotqa": "HotpotQA subset"}


def _ms(x):
    x = np.asarray(x, dtype=float)
    return f"{x.mean():.3f} ± {x.std(ddof=1):.3f}" if len(x) > 1 else f"{x.mean():.3f}"


def _label(s, z):
    return NAME[s] if s == C.BASELINE else f"{NAME[s]} ({z})"


def _order(df):
    keys = [(C.BASELINE, 0)] + [(s, z) for s in C.STRATEGIES for z in C.CHUNK_SIZES]
    return [k for k in keys if ((df.strategy == k[0]) & (df.chunk_size == k[1])).any()]


def table1(res):
    rows = []
    for ds, g in res.groupby("dataset"):
        for s, z in _order(g):
            sub = g[(g.strategy == s) & (g.chunk_size == z)]
            row = {"Dataset": DS_NAME[ds], "Strategy": _label(s, z) + (" [baseline]" if s == C.BASELINE else "")}
            for m, lab in (("recall@5", "Recall@5"), ("recall@10", "Recall@10"), ("mrr@10", "MRR@10"), ("ndcg@10", "NDCG@10")):
                row[lab] = _ms(sub[sub.metric == m].sort_values("seed")["value"])
            rows.append(row)
    return pd.DataFrame(rows)


def table2(cost):
    rows = []
    for ds, g in cost.groupby("dataset"):
        for s, z in _order(g):
            sub = g[(g.strategy == s) & (g.chunk_size == z)]
            rows.append({"Dataset": DS_NAME[ds], "Strategy": _label(s, z) + (" [baseline]" if s == C.BASELINE else ""),
                         "Chunks": f"{sub.n_chunks.mean():,.0f}", "Mean tokens/chunk": f"{sub.mean_chunk_tokens.mean():.0f}",
                         "Chunking time (s)": f"{sub.chunk_seconds.mean():.2f}",
                         "Embedding time (s)": f"{sub.embed_seconds.mean():.2f}",
                         "Indexing time (s)": f"{sub.index_seconds.mean():.2f}"})
    return pd.DataFrame(rows)


def table3_ci(pq):
    rows = []
    for ds, g in pq.groupby("dataset"):
        for s, z in _order(g):
            v = g[(g.strategy == s) & (g.chunk_size == z)]["ndcg@10"]
            lo, hi = bootstrap_ci(v, C.BOOTSTRAP)
            rows.append({"Dataset": DS_NAME[ds], "Strategy": _label(s, z), "NDCG@10": f"{v.mean():.3f}",
                         "95% bootstrap CI": f"[{lo:.3f}, {hi:.3f}]", "Queries (pooled over seeds)": len(v)})
    return pd.DataFrame(rows)


def figure1(res, path):
    dsets = list(res.dataset.unique())
    fig, axes = plt.subplots(1, len(dsets), figsize=(5.5 * len(dsets), 4), squeeze=False)
    for ax, ds in zip(axes[0], dsets):
        g = res[(res.dataset == ds) & (res.metric == "ndcg@10")]
        w = 0.8 / len(C.CHUNK_SIZES)
        for i, z in enumerate(C.CHUNK_SIZES):
            vals, errs = [], []
            for s in C.STRATEGIES:
                v = g[(g.strategy == s) & (g.chunk_size == z)]["value"]
                vals.append(v.mean()); errs.append(v.std(ddof=1) if len(v) > 1 else 0)
            ax.bar(np.arange(len(C.STRATEGIES)) + i * w - 0.4 + w / 2, vals, w, yerr=errs, capsize=3,
                   color=[COL[s] for s in C.STRATEGIES], alpha=0.55 if i == 0 else 1.0,
                   edgecolor="white", label=f"{z} tokens")
        b = g[g.strategy == C.BASELINE]["value"].mean()
        ax.axhline(b, color=COL["whole_doc"], ls="--", lw=1, label="No chunking (baseline)")
        ax.set_xticks(range(len(C.STRATEGIES)))
        ax.set_xticklabels([NAME[s] for s in C.STRATEGIES])
        ax.set_xlabel("Chunking strategy (light = %d tokens, dark = %d tokens)" % tuple(C.CHUNK_SIZES))
        ax.set_ylabel("NDCG@10 (mean ± std over seeds)")
        ax.set_title(DS_NAME[ds])
        ax.legend(frameon=False, fontsize=8)
        ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(); fig.savefig(path, dpi=200); plt.close(fig)


def figure2(res, path):
    dsets = list(res.dataset.unique())
    fig, axes = plt.subplots(1, len(dsets), figsize=(5.5 * len(dsets), 4), squeeze=False)
    for ax, ds in zip(axes[0], dsets):
        g = res[res.dataset == ds]
        for s, z in _order(g):
            sub = g[(g.strategy == s) & (g.chunk_size == z) & g.metric.str.startswith("recall@")].copy()
            sub["k"] = sub.metric.str.split("@").str[1].astype(int)
            c = sub.groupby("k")["value"].mean()
            ax.plot(c.index, c.values, color=COL[s], lw=1.6, ls="-" if (s == C.BASELINE or z == C.CHUNK_SIZES[-1]) else "--",
                    label=_label(s, z))
        ax.set_xlabel("Cutoff K (number of retrieved documents)")
        ax.set_ylabel("Recall@K")
        ax.set_xticks([1, 5, 10, 15, 20])
        ax.set_title(DS_NAME[ds])
        ax.legend(frameon=False, fontsize=7)
        ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(); fig.savefig(path, dpi=200); plt.close(fig)


def figure3_hotpot(pq, path):
    g = pq[(pq.dataset == "hotpotqa") & (pq.qtype != "")]
    if g.empty:
        return
    types = sorted(g.qtype.unique())
    keys = _order(g)
    fig, ax = plt.subplots(figsize=(7, 4))
    w = 0.8 / len(types)
    for i, t in enumerate(types):
        vals = [g[(g.qtype == t) & (g.strategy == s) & (g.chunk_size == z)]["ndcg@10"].mean() for s, z in keys]
        ax.bar(np.arange(len(keys)) + i * w - 0.4 + w / 2, vals, w, label=t)
    ax.set_xticks(range(len(keys)))
    ax.set_xticklabels([_label(s, z).replace(" (", "\n(") for s, z in keys], fontsize=7)
    ax.set_xlabel("Strategy (chunk size in tokens)")
    ax.set_ylabel("NDCG@10 (mean over seeds)")
    ax.legend(title="HotpotQA question type", frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(); fig.savefig(path, dpi=200); plt.close(fig)


def failure_cases(pq, n=5):
    """Queries (seed 0) where the best and worst chunking configuration differ most in NDCG@10."""
    rows = []
    for ds, g in pq[pq.seed == 0].groupby("dataset"):
        wide = g.assign(cfg=g.strategy + "_" + g.chunk_size.astype(str)).pivot(index="qid", columns="cfg", values="ndcg@10")
        gap = (wide.max(axis=1) - wide.min(axis=1)).sort_values(ascending=False)
        for qid in gap.index[: (3 if ds == "nfcorpus" else 2)]:
            best, worst = wide.loc[qid].idxmax(), wide.loc[qid].idxmin()
            sel = g[g.qid == qid]
            get = lambda c: sel[(sel.strategy + "_" + sel.chunk_size.astype(str)) == c].iloc[0]
            rows.append({"dataset": ds, "qid": qid, "best_config": best, "best_ndcg@10": wide.loc[qid, best],
                         "worst_config": worst, "worst_ndcg@10": wide.loc[qid, worst],
                         "n_relevant": int(sel.n_relevant.iloc[0]), "relevant_ids": sel.relevant.iloc[0],
                         "top10_best": get(best).top10, "top10_worst": get(worst).top10})
    return pd.DataFrame(rows).head(n)


def doc_type_summary(res, docstats):
    rows = []
    for ds, g in res[res.metric == "ndcg@10"].groupby("dataset"):
        agg = g.groupby(["strategy", "chunk_size"])["value"].mean()
        chunked = agg.drop(index=(C.BASELINE, 0), errors="ignore")
        (bs, bz), bv = chunked.idxmax(), chunked.max()
        base = agg.get((C.BASELINE, 0), np.nan)
        fixed_best = agg[[i for i in agg.index if i[0] == "fixed"]].max()
        d = docstats[docstats.dataset == ds].mean(numeric_only=True)
        rows.append({"dataset": DS_NAME[ds], "mean_doc_tokens": round(d.mean_doc_tokens), "median_doc_tokens": round(d.median_doc_tokens),
                     **{f"pct_docs_over_{s}_tokens": round(d[f"pct_docs_over_{s}"], 1) for s in C.CHUNK_SIZES},
                     "best_chunked_config": f"{bs} ({bz})", "best_chunked_ndcg@10": round(bv, 4),
                     "no_chunking_ndcg@10": round(base, 4), "best_fixed_ndcg@10": round(fixed_best, 4),
                     "best_minus_no_chunking": round(bv - base, 4), "best_minus_best_fixed": round(bv - fixed_best, 4)})
    return pd.DataFrame(rows)


def main():
    R = C.RESULTS
    res = pd.read_csv(R / "results.csv")
    cost = pd.read_csv(R / "index_cost.csv")
    pq = pd.read_csv(R / "per_query.csv.gz", keep_default_na=False)
    pq["seed"] = pq["seed"].astype(int)
    docstats = pd.read_csv(R / "doc_stats.csv")

    t1, t2, t3 = table1(res), table2(cost), table3_ci(pq)
    t1.to_csv(R / "T1_quality.csv", index=False)
    t2.to_csv(R / "T2_cost.csv", index=False)
    t3.to_csv(R / "T3_ndcg_ci.csv", index=False)
    summ = doc_type_summary(res, docstats)
    summ.to_csv(R / "doc_type_summary.csv", index=False)
    failure_cases(pq).to_csv(R / "failure_cases.csv", index=False)
    (R / "tables.md").write_text(
        "## T1: retrieval quality (mean ± std over seeds; NFCorpus is deterministic, single run)\n\n" + t1.to_markdown(index=False)
        + "\n\n## T2: chunks and indexing time\n\n" + t2.to_markdown(index=False)
        + "\n\n## T3: NDCG@10 bootstrap CI\n\n" + t3.to_markdown(index=False)
        + "\n\n## Document type summary\n\n" + summ.to_markdown(index=False) + "\n", encoding="utf-8")
    figure1(res, R / "F1_ndcg_by_strategy.png")
    figure2(res, R / "F2_recall_curves.png")
    figure3_hotpot(pq, R / "F3_hotpot_by_qtype.png")
    print(t1.to_string(index=False)); print(); print(t2.to_string(index=False)); print(); print(summ.T.to_string())


if __name__ == "__main__":
    main()

## 3. Self-test the metrics, then run the full experiment

In [ ]:
!python -m p1.metrics

This is the main step. It prints one line per configuration (chunks, Recall@5/10, MRR, NDCG@10, indexing time)
for NFCorpus (once, deterministic) and HotpotQA (3 seeds = 3 different question samples).

In [ ]:
!python -m p1.run_experiment

## 4. Build tables (T1, T2), figures (F1, F2) and failure cases

In [ ]:
!python -m p1.report
import pandas as pd
from IPython.display import display, Image
pd.set_option('display.width', 200)
print('T1: retrieval quality'); display(pd.read_csv('results/T1_quality.csv'))
print('T2: chunks and indexing time'); display(pd.read_csv('results/T2_cost.csv'))
print('NDCG@10 bootstrap CI'); display(pd.read_csv('results/T3_ndcg_ci.csv'))
print('Document type summary'); display(pd.read_csv('results/doc_type_summary.csv').T)
display(Image('results/F1_ndcg_by_strategy.png'))
display(Image('results/F2_recall_curves.png'))
display(Image('results/F3_hotpot_by_qtype.png'))

## 5. Download everything (results.csv, tables, figures, logs)

In [ ]:
!cd /content/p1proj && zip -qr /content/p1_variant_results.zip results p1
from google.colab import files
files.download('/content/p1_variant_results.zip')

## What to put in the report
- **T1, T2, F1, F2** are in `results/` (`T1_quality.csv`, `T2_cost.csv`, `F1_ndcg_by_strategy.png`, `F2_recall_curves.png`).
- `doc_type_summary.csv` gives document length statistics next to the best strategy per dataset: use it for the
  "which strategy suits which document type" note.
- `failure_cases.csv` lists the queries where the best and worst configuration differ most: pick 5 and explain them.
- `results.csv` holds every metric per seed (submit this). `run_info.json` records versions, GPU and settings.
- Report numbers exactly as printed. HotpotQA paragraphs are short, so chunking may change little there; say so if that is what you see.